In [3]:
import sys
from pathlib import Path

# Add the project root to Python path so we can import src

# Get current working directory
cwd = Path().resolve()

# If we're in the notebooks directory, go up one level
# Otherwise, assume we're already in the project root
if cwd.name == "notebooks":
	project_root = cwd.parent
else:
	project_root = cwd

if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

In [4]:
from typing import Any, List, Dict
from typing_extensions import Annotated

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.tools.base import InjectedToolCallId
from langchain_core.messages import ToolMessage
from langgraph.graph import MessagesState
from langgraph.types import Command

from src.rag.retrieval.index import retrieve_context
from src.rag.retrieval.utils import prepare_prompt_and_invoke_llm

In [5]:
# create a custom agent that extends the MessageState to store citations
class CustomAgentState(MessagesState):
    """
    Extended agent state with citations tracking and guardrail status.
    """
    # citations will accumulate across tool calls
    citations: Annotated[List[Dict[str, Any]], lambda x, y: x + y] = []

In [6]:
def create_rag_tool(project_id: str):
    """
    Create a RAG search tool bound to a specific project.
    """
    
    @tool
    def rag_search(
        query: str,
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        """
        Search through project documents using RAG (Retrieval-Augmented Generation).
        This tool retrieves relevant context from the current project's documents based on the query.
        
        Args:
            query: The search query or question to find relevant information
            tool_call_id: Injected tool call ID for message tracking
            
        Returns:
            A Command object with updated messages and citations
        """
        try:
            # Retrieve context using the existing RAG pipeline
            texts, images, tables, citations = retrieve_context(project_id, query)
            
            # If no context found, return a message
            if not texts:
                return Command(
                    update={
                        "messages": [
                            ToolMessage(
                                "No relevant information found in the project documents for this query.",
                                tool_call_id=tool_call_id
                            )
                        ]
                    }
                )
                
            # Prepare the response using the existing LLM preparation function
            response = prepare_prompt_and_invoke_llm(
                user_query=query,
                texts=texts,
                images=images,
                tables=tables
            )
            
            return Command(
                update={
                    # Update message history
                    "messages": [
                        ToolMessage(
                            content=response,
                            tool_call_id=tool_call_id
                        )
                    ],
                    # Update citations in state - these accumulate!
                    "citations": citations
                }
            )
            
        except Exception as e:
            return Command(
                update={
                    "messages": [
                        ToolMessage(
                            f"Error retrieving information: {str(e)}",
                            tool_call_id=tool_call_id
                        )
                    ]
                }
            )

    return rag_search

In [7]:
def create_rag_agent(
    project_id: str,
    model: str = "gpt-4o",
):
    """
    Create an agent with RAG tool for a specific project
    """
    # Create tools list with project-specific RAG tool
    tools = [create_rag_tool(project_id)]
    
    # Get the system prompt with optional chat history
    system_prompt = """You are a helpful AI assistant with access to a RAG (Retrieval-Augmented Generation) tool that searches project-specific documents.

        For every user question:

        1. Do not assume any question is purely conceptual or general.  
        2. Use the `rag_search` tool immediately with a clear and relevant query derived from the user's question. 
        3. Use the chat history to understand the context and references in the current question. 
        4. Carefully review the retrieved documents and base your entire answer on the RAG results.  
        5. If the retrieved information fully answers the user's question, respond clearly and completely using that information.  
        6. If the retrieved information is insufficient or incomplete, explicitly state that and provide helpful suggestions or guidance based on what you found.  
        7. Always present answers in a clear, well-structured, and conversational manner.

        **Make sure to call the rag_search tool correctly**
        **Never answer without first querying the RAG tool. This ensures every response is grounded in project-specific context and documentation.**
        """
    
    # Create the base agent
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt=system_prompt,
        state_schema=CustomAgentState
    )
    
    
    return agent

In [8]:
project_id = "6d090d75-7c7c-428c-bba8-258cf3f45d2d"
rag_agent = create_rag_agent(project_id=project_id, model="gpt-4o")

In [9]:
inputs = {"messages": [{"role": "user", "content": "What are the two types of sleep?"}]}

result = rag_agent.invoke(inputs)

Fetching project settings...


In [10]:
result["messages"][-1]

AIMessage(content="It seems there is an issue retrieving information from the documents at the moment. However, I can provide you with general knowledge on this topic:\n\nThe two main types of sleep are:\n\n1. **REM (Rapid Eye Movement) Sleep**: This is the sleep phase where dreaming most often occurs. It is characterized by rapid movement of the eyes, increased brain activity, and vivid dreams. REM sleep is essential for cognitive functions like memory and learning.\n\n2. **NREM (Non-Rapid Eye Movement) Sleep**: This type of sleep does not involve rapid eye movements and is divided into three stages, each progressively getting deeper. NREM sleep is important for physical restoration, growth, and immune system strengthening.\n\nIf you need more specific information or if there's a different context related to your question, please let me know!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 398, 'total_tokens': 563, '

In [11]:
result

{'messages': [HumanMessage(content='What are the two types of sleep?', additional_kwargs={}, response_metadata={}, id='a2d9dd46-5486-4dcc-a4e2-73268b3dee8f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 339, 'total_tokens': 357, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_b558dd220d', 'id': 'chatcmpl-DLlr8GNCMl5QBDQ379SzqHBzyT3f1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0f7b-28d0-7b22-aff8-8f5c73e92eca-0', tool_calls=[{'name': 'rag_search', 'args': {'query': 'two types of sleep'}, 'id': 'call_RW2FdSvW8YdRc0oRG8XIEB4i', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'